In [1]:
# Recharge automatiquement src/transformers.py quand il est modifié
%load_ext autoreload
%autoreload 2

# Rend le dossier src/ (un niveau au-dessus de notebooks/) importable
import sys
sys.path.append("..")
from src.transformers import GroupMedianImputer, ModeImputer, Log1pTransformer, OrdinalMapper

import pandas as pd
import numpy as np
from src.preprocessing import clean, remove_outliers, add_features

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [2]:

train = pd.read_csv("../data/train.csv")
test = pd.read_csv("../data/test.csv")
test_ids = test["Id"]  

train.shape, test.shape

((1460, 81), (1459, 80))

In [3]:
train = remove_outliers(train)

y = np.log1p(train["SalePrice"])

# Features uniquement : on retire l'identifiant et la cible
X_train = train.drop(columns=["Id", "SalePrice"])
X_test  = test.drop(columns=["Id"])

# Concaténation train + test ; n_train sert à re-séparer plus tard
n_train = len(X_train)
all_X = pd.concat([X_train, X_test], axis=0, ignore_index=True)
all_X.shape 

(2917, 79)

In [4]:
cleaned = clean(all_X)
cleaned_featured = add_features(cleaned)

In [5]:
X_train = cleaned_featured.iloc[:n_train].copy()
X_test = cleaned_featured.iloc[n_train:].copy()

In [6]:
print(X_train.shape) 
print(X_test.shape) 
print(y.shape) 

(1458, 80)
(1459, 80)
(1458,)


In [7]:
na = X_train.isna().sum()
print(na[na > 0])

LotFrontage     259
MasVnrType        5
BsmtExposure      1
BsmtFinType2      1
Electrical        1
dtype: int64


In [8]:
# Colonnes passées en log1p (section 4 + décisions de la section 6)
cols_log = [
    # Surfaces à longue traîne
    "LotArea", "LotFrontage", "GrLivArea", "1stFlrSF", "TotalBsmtSF", "BsmtUnfSF",
    # Beaucoup de zéros mais vraie quantité
    "2ndFlrSF", "WoodDeckSF", "OpenPorchSF", "MasVnrArea", "BsmtFinSF1",
    # Presque vides : log1p plutôt que TotalPorchSF (rejetée)
    "MiscVal", "EnclosedPorch", "ScreenPorch", "3SsnPorch",
    # Feature engineering
    "TotalSF",
]

In [9]:
# Échelle de qualité commune (None = absent)
QUAL = {"None": 0, "Po": 1, "Fa": 2, "TA": 3, "Gd": 4, "Ex": 5}

# Colonnes qui utilisent l'échelle de qualité Po → Ex
cols_qual = [
    "ExterQual", "ExterCond",
    "BsmtQual", "BsmtCond",
    "HeatingQC", "KitchenQual", "FireplaceQu",
    "GarageQual", "GarageCond",
]

mapping_qual = {col: QUAL for col in cols_qual}

# Échelle partagée par les deux types de finition du sous-sol
FINTYPE = {"None": 0, "Unf": 1, "LwQ": 2, "Rec": 3, "BLQ": 4, "ALQ": 5, "GLQ": 6}

# Ordinales avec leur propre échelle (du pire au meilleur)
mapping_other = {
    "BsmtExposure": {"None": 0, "No": 1, "Mn": 2, "Av": 3, "Gd": 4},
    "BsmtFinType1": FINTYPE,
    "BsmtFinType2": FINTYPE,
    "GarageFinish": {"None": 0, "Unf": 1, "RFn": 2, "Fin": 3},
    "Functional":   {"Sev": 0, "Maj2": 1, "Maj1": 2, "Mod": 3, "Min2": 4, "Min1": 5, "Typ": 6},
    "PavedDrive":   {"N": 0, "P": 1, "Y": 2},
    "CentralAir":   {"N": 0, "Y": 1},
    "Street":       {"Grvl": 0, "Pave": 1},
}

# Les 17 ordinales : qualité + échelles propres
mapping_ord = mapping_other | mapping_qual

In [10]:
# Nominales = colonnes texte qui ne sont pas ordinales (→ one-hot)
cols_cate = X_train.select_dtypes(exclude="number").columns.tolist()
cols_nom = [c for c in cols_cate if c not in mapping_ord]

In [11]:
# Les étapes du Pipeline (non entraînées) : 17 imputers, log1p, encodage ordinal, one-hot, standardisation ; fit sur le train uniquement
imp_lotfrontage    = GroupMedianImputer()                                         # médiane par Neighborhood
mode_masvnrtype    = ModeImputer()                                                # mode hors "None"

imp_garage_area    = GroupMedianImputer(target_col="GarageArea", group_col="GarageType")
imp_garage_cars    = GroupMedianImputer(target_col="GarageCars", group_col="GarageType")
mode_garage_finish = ModeImputer(target_col="GarageFinish")
mode_garage_qual   = ModeImputer(target_col="GarageQual")
mode_garage_cond   = ModeImputer(target_col="GarageCond")

mode_bsmt_cond     = ModeImputer(target_col="BsmtCond")
mode_bsmt_qual     = ModeImputer(target_col="BsmtQual")
mode_bsmt_exposure = ModeImputer(target_col="BsmtExposure")
mode_bsmt_fintype2 = ModeImputer(target_col="BsmtFinType2", exclude_value=["None", "Unf"])  # type aménagé le plus fréquent

mode_mszoning      = ModeImputer(target_col="MSZoning")
mode_exterior1st   = ModeImputer(target_col="Exterior1st")
mode_exterior2nd   = ModeImputer(target_col="Exterior2nd")
mode_electrical    = ModeImputer(target_col="Electrical")
mode_kitchenqual   = ModeImputer(target_col="KitchenQual")
mode_saletype      = ModeImputer(target_col="SaleType")

log_transfo        = Log1pTransformer(cols=cols_log)
ordinal_mapper     = OrdinalMapper(mapping=mapping_ord)
encodeur           = ColumnTransformer(
    transformers=[
        # One-hot des 25 nominales ; catégorie inconnue du test → 0 partout
        ("ohe", OneHotEncoder(handle_unknown="ignore", drop=None, sparse_output=False), cols_nom),
    ],
    remainder="passthrough",              # les 55 autres colonnes passent telles quelles
    verbose_feature_names_out=False,      # noms sans préfixe : GrLivArea, Neighborhood_NAmes
).set_output(transform="pandas")
scaleur            = StandardScaler().set_output(transform="pandas")   # moyenne 0, écart-type 1 (appris sur le train) : pénalité du Lasso équitable

In [12]:
# Pipeline (21 étapes : imputations, log1p, encodage ordinal, one-hot, standardisation) : fit sur le train uniquement, puis appliqué au train et au test
pipe = Pipeline([
    ("lotfrontage",    imp_lotfrontage),
    ("masvnrtype",     mode_masvnrtype),
    ("garage_area",    imp_garage_area),
    ("garage_cars",    imp_garage_cars),
    ("garage_finish",  mode_garage_finish),
    ("garage_qual",    mode_garage_qual),
    ("garage_cond",    mode_garage_cond),
    ("bsmt_cond",      mode_bsmt_cond),
    ("bsmt_qual",      mode_bsmt_qual),
    ("bsmt_exposure",  mode_bsmt_exposure),
    ("bsmt_fintype2",  mode_bsmt_fintype2),
    ("mszoning",       mode_mszoning),
    ("exterior1st",    mode_exterior1st),
    ("exterior2nd",    mode_exterior2nd),
    ("electrical",     mode_electrical),
    ("kitchenqual",    mode_kitchenqual),
    ("saletype",       mode_saletype),
    ("log_of_cols",    log_transfo),
    ("ordinal_map",    ordinal_mapper),
    ("one_hot_encod",  encodeur),
    ("standard_scal",  scaleur)
])

pipe.fit(X_train)
X_train_prep = pipe.transform(X_train)
X_test_prep  = pipe.transform(X_test)

In [13]:
# Contrôle de sortie du Pipeline
na_train = X_train_prep.isna().sum().sum()
na_test  = X_test_prep.isna().sum().sum()
print("NaN train / test               :", na_train, "/", na_test)                       # attendu : 0 / 0

print("Formes train / test            :", X_train_prep.shape, X_test_prep.shape)        # attendu : (1458, n) et (1459, n)
print("Mêmes colonnes, même ordre     :", list(X_train_prep.columns) == list(X_test_prep.columns))  # attendu : True
print("Colonnes créées par le one-hot :", X_train_prep.shape[1] - (80 - len(cols_nom)))

print("Type train / test              :", type(X_train_prep).__name__, "/", type(X_test_prep).__name__)  # attendu : DataFrame

cols_texte_restantes = X_train_prep.select_dtypes(exclude="number").columns.tolist()
print("Colonnes texte restantes       :", cols_texte_restantes)                         # attendu : []

# Log1p : vérifié AVANT le scaler (pipe[:-1] = toutes les étapes sauf la dernière)
X_train_avant_scaler = pipe[:-1].transform(X_train)
print("Max des colonnes en log1p, avant scaler (attendu < 13) :", X_train_avant_scaler[cols_log].max().max().round(2))

# Standardisation : sur le train, chaque colonne doit avoir moyenne 0 et écart-type 1
print("Train : |moyenne| max          :", X_train_prep.mean().abs().max().round(6))             # attendu : ≈ 0
print("Train : écart-type min / max   :", X_train_prep.std(ddof=0).min().round(4), "/",
      X_train_prep.std(ddof=0).max().round(4))                                                # attendu : 1 / 1

# Sur le test : proche de 0 et 1, mais pas exactement (centré avec la moyenne du train)
cols_exemple = ["GrLivArea", "OverallQual", "CentralAir"]
print(pd.DataFrame({
    "moy train": X_train_prep[cols_exemple].mean(),
    "moy test":  X_test_prep[cols_exemple].mean(),
    "std train": X_train_prep[cols_exemple].std(ddof=0),
    "std test":  X_test_prep[cols_exemple].std(ddof=0),
}).round(3))

NaN train / test               : 0 / 0
Formes train / test            : (1458, 253) (1459, 253)
Mêmes colonnes, même ordre     : True
Colonnes créées par le one-hot : 198
Type train / test              : DataFrame / DataFrame
Colonnes texte restantes       : []
Max des colonnes en log1p, avant scaler (attendu < 13) : 12.28
Train : |moyenne| max          : 0.0
Train : écart-type min / max   : 1.0 / 1.0
             moy train  moy test  std train  std test
GrLivArea         -0.0    -0.037        1.0     0.957
OverallQual       -0.0    -0.011        1.0     1.044
CentralAir         0.0    -0.016        1.0     1.028
